# 107 - Voronoi ピボット（セントロイド）エクスポート

3 つのモデルについて pyconjp 23K 件の画像埋め込みで k-means ピボット（256 個）を学習し、numpy (.npy) と JSON (.json) で出力する。

| モデル | 次元 | 画像数 | pyconjp DB |
|--------|:---:|:---:|--------|
| SigLIP 2 Base | 768 | ~23,400 | `pyconjp_image_search.duckdb` |
| SigLIP 2 Large | 1024 | ~23,400 | `pyconjp_image_search_siglip2_large.duckdb` |
| CLIP-L | 768 | ~23,400 | `pyconjp_image_search_clip.duckdb` |

### 出力ファイル
- `data/voronoi_pivots_{model_short}.npy` — numpy 形式 (shape: [256, dim])
- `data/voronoi_pivots_{model_short}.json` — JSON 形式 (メタデータ付き)

### 汎化性能
NB106 で、Train セットのみでピボットを学習しても未知データに対して精度劣化がないことを確認済み。
23K 件でのピボット学習は実運用に十分な汎化性能を持つ。

In [5]:
import json
from datetime import datetime
from pathlib import Path

import duckdb
import numpy as np
from sklearn.cluster import MiniBatchKMeans

DB_PATH = Path("../data/images.duckdb")
PYCONJP_DB_DIR = Path("/home/terapyon/dev/vibe-coding/pyconjp-image-search")
OUTPUT_DIR = Path("../data")
RANDOM_STATE = 42
N_PIVOTS = 256

# モデル定義（全モデル pyconjp 23K 件 + カタログ 378 件、256 ピボット）
MODELS = [
    {
        "short_name": "siglip2_base",
        "model_name": "google/siglip2-base-patch16-224",
        "dim": 768,
        "catalog_table": "image_embeddings_768",
        "pyconjp_db": PYCONJP_DB_DIR / "pyconjp_image_search.duckdb",
    },
    {
        "short_name": "siglip2_large",
        "model_name": "google/siglip2-large-patch16-256",
        "dim": 1024,
        "catalog_table": "image_embeddings_1024",
        "pyconjp_db": PYCONJP_DB_DIR / "pyconjp_image_search_siglip2_large.duckdb",
    },
    {
        "short_name": "clip_large",
        "model_name": "openai/clip-vit-large-patch14",
        "dim": 768,
        "catalog_table": "image_embeddings_768",
        "pyconjp_db": PYCONJP_DB_DIR / "pyconjp_image_search_clip.duckdb",
    },
]

print(f"出力先: {OUTPUT_DIR}")
print(f"ピボット数: {N_PIVOTS}")
for m in MODELS:
    print(f"  {m['short_name']}: {m['model_name']} ({m['dim']}D)")
    print(f"    pyconjp DB: {m['pyconjp_db'].name}")

出力先: ../data
ピボット数: 256
  siglip2_base: google/siglip2-base-patch16-224 (768D)
    pyconjp DB: pyconjp_image_search.duckdb
  siglip2_large: google/siglip2-large-patch16-256 (1024D)
    pyconjp DB: pyconjp_image_search_siglip2_large.duckdb
  clip_large: openai/clip-vit-large-patch14 (768D)
    pyconjp DB: pyconjp_image_search_clip.duckdb


## ピボット学習とエクスポート

In [6]:
def load_embeddings(model_cfg):
    """カタログ DB + pyconjp DB から埋め込みを読み込み、L2正規化して返す。"""
    # カタログ DB (378 件)
    conn = duckdb.connect(str(DB_PATH), read_only=True)
    conn.execute("LOAD vss;")
    rows = conn.execute(f"""
        SELECT id, embedding FROM {model_cfg['catalog_table']}
        WHERE model_name = ?
        ORDER BY id
    """, [model_cfg["model_name"]]).fetchall()
    conn.close()

    ids = [r[0] for r in rows]
    embs = [[float(x) for x in r[1]] for r in rows]

    # pyconjp DB (~23K 件)
    conn_pj = duckdb.connect(str(model_cfg["pyconjp_db"]), read_only=True)
    pj_rows = conn_pj.execute("""
        SELECT i.id, e.embedding
        FROM image_embeddings e
        JOIN images i ON e.image_id = i.id
        WHERE e.model_name = ?
        ORDER BY i.id
    """, [model_cfg["model_name"]]).fetchall()
    conn_pj.close()

    ids += [r[0] for r in pj_rows]
    embs += [[float(x) for x in r[1]] for r in pj_rows]

    embeddings = np.array(embs, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings_normed = embeddings / norms

    return ids, embeddings_normed


def train_and_export(model_cfg):
    """ピボットを学習し、numpy と JSON で出力する。"""
    short = model_cfg["short_name"]
    dim = model_cfg["dim"]

    print(f"\n{'='*60}")
    print(f"{short}: {model_cfg['model_name']}")
    print(f"{'='*60}")

    # 埋め込み読み込み
    ids, embeddings_normed = load_embeddings(model_cfg)
    n_images = len(ids)
    print(f"  画像数: {n_images}, 次元: {embeddings_normed.shape[1]}")

    # k-means
    kmeans = MiniBatchKMeans(
        n_clusters=N_PIVOTS, random_state=RANDOM_STATE,
        batch_size=2048, n_init=3
    )
    kmeans.fit(embeddings_normed)
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

    pivot_sizes = np.bincount(kmeans.labels_, minlength=N_PIVOTS)
    print(f"  ピボット数: {N_PIVOTS}")
    print(f"  ピボットサイズ: min={pivot_sizes.min()}, max={pivot_sizes.max()}, "
          f"mean={pivot_sizes.mean():.1f}, median={np.median(pivot_sizes):.0f}")

    # numpy 出力
    npy_path = OUTPUT_DIR / f"voronoi_pivots_{short}.npy"
    np.save(npy_path, centroids_normed)
    print(f"  numpy: {npy_path.name} ({npy_path.stat().st_size / 1024:.1f} KB)")

    # JSON 出力
    json_data = {
        "model_name": model_cfg["model_name"],
        "short_name": short,
        "n_pivots": N_PIVOTS,
        "dim": dim,
        "n_images_trained": n_images,
        "created_at": datetime.now().isoformat(),
        "centroids": centroids_normed.tolist(),
    }
    json_path = OUTPUT_DIR / f"voronoi_pivots_{short}.json"
    with open(json_path, "w") as f:
        json.dump(json_data, f)
    print(f"  JSON:  {json_path.name} ({json_path.stat().st_size / 1024:.1f} KB)")

    return centroids_normed

print("関数定義完了")

関数定義完了


In [7]:
# 全モデルで学習・エクスポート
all_centroids = {}
for model_cfg in MODELS:
    centroids = train_and_export(model_cfg)
    all_centroids[model_cfg["short_name"]] = centroids

print(f"\n\n{'='*60}")
print("エクスポート完了")
print(f"{'='*60}")
for short, c in all_centroids.items():
    print(f"  {short}: shape={c.shape}")


siglip2_base: google/siglip2-base-patch16-224


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  画像数: 23464, 次元: 768
  ピボット数: 256
  ピボットサイズ: min=1, max=310, mean=91.7, median=86
  numpy: voronoi_pivots_siglip2_base.npy (768.1 KB)
  JSON:  voronoi_pivots_siglip2_base.json (4290.1 KB)

siglip2_large: google/siglip2-large-patch16-256


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  画像数: 23464, 次元: 1024
  ピボット数: 256
  ピボットサイズ: min=1, max=344, mean=91.7, median=90
  numpy: voronoi_pivots_siglip2_large.npy (1024.1 KB)
  JSON:  voronoi_pivots_siglip2_large.json (5727.2 KB)

clip_large: openai/clip-vit-large-patch14


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  画像数: 23464, 次元: 768
  ピボット数: 256
  ピボットサイズ: min=1, max=322, mean=91.7, median=87
  numpy: voronoi_pivots_clip_large.npy (768.1 KB)
  JSON:  voronoi_pivots_clip_large.json (4274.9 KB)


エクスポート完了
  siglip2_base: shape=(256, 768)
  siglip2_large: shape=(256, 1024)
  clip_large: shape=(256, 768)


## 検証: エクスポートデータの読み込みテスト

In [8]:
# numpy / JSON 両方の読み込みテスト + 一致確認
for model_cfg in MODELS:
    short = model_cfg["short_name"]

    # numpy
    npy_path = OUTPUT_DIR / f"voronoi_pivots_{short}.npy"
    centroids_npy = np.load(npy_path)

    # JSON
    json_path = OUTPUT_DIR / f"voronoi_pivots_{short}.json"
    with open(json_path) as f:
        data = json.load(f)
    centroids_json = np.array(data["centroids"], dtype=np.float32)

    # 一致確認
    max_diff = np.abs(centroids_npy - centroids_json).max()
    norms_check = np.linalg.norm(centroids_npy, axis=1)

    print(f"{short}:")
    print(f"  numpy shape: {centroids_npy.shape}, dtype: {centroids_npy.dtype}")
    print(f"  JSON  shape: {centroids_json.shape}")
    print(f"  max diff (npy vs json): {max_diff:.2e}")
    print(f"  L2 norm range: [{norms_check.min():.6f}, {norms_check.max():.6f}]")
    print(f"  JSON metadata: model={data['model_name']}, n_images={data['n_images_trained']}, pivots={data['n_pivots']}")
    print()

siglip2_base:
  numpy shape: (256, 768), dtype: float32
  JSON  shape: (256, 768)
  max diff (npy vs json): 0.00e+00
  L2 norm range: [1.000000, 1.000000]
  JSON metadata: model=google/siglip2-base-patch16-224, n_images=23464, pivots=256

siglip2_large:
  numpy shape: (256, 1024), dtype: float32
  JSON  shape: (256, 1024)
  max diff (npy vs json): 0.00e+00
  L2 norm range: [1.000000, 1.000000]
  JSON metadata: model=google/siglip2-large-patch16-256, n_images=23464, pivots=256

clip_large:
  numpy shape: (256, 768), dtype: float32
  JSON  shape: (256, 768)
  max diff (npy vs json): 0.00e+00
  L2 norm range: [1.000000, 1.000000]
  JSON metadata: model=openai/clip-vit-large-patch14, n_images=23464, pivots=256



## まとめ

### やったこと

3 つの画像埋め込みモデルについて、Voronoi 分割用の **k-means ピボット（セントロイド）** を学習し、numpy と JSON の 2 形式でエクスポートした。

### データソース

各モデルのカタログ画像 378 件（`images.duckdb`）と pyconjp アーカイブ 23,086 件の合計 **23,464 件** の画像埋め込み（L2 正規化済み）を使用。

| モデル | pyconjp DB |
|--------|------------|
| SigLIP 2 Base (`google/siglip2-base-patch16-224`) | `pyconjp_image_search.duckdb` |
| SigLIP 2 Large (`google/siglip2-large-patch16-256`) | `pyconjp_image_search_siglip2_large.duckdb` |
| CLIP-L (`openai/clip-vit-large-patch14`) | `pyconjp_image_search_clip.duckdb` |

### 学習パラメータ

- **アルゴリズム**: MiniBatchKMeans（scikit-learn）
- **ピボット数**: 256（全モデル共通）
- **batch_size**: 2048, **n_init**: 3, **random_state**: 42
- セントロイドは学習後に L2 正規化（コサイン類似度 = ドット積として使用可能）

### 出力ファイル

| ファイル | サイズ |
|----------|:---:|
| `voronoi_pivots_siglip2_base.npy` | 768 KB |
| `voronoi_pivots_siglip2_base.json` | 4,290 KB |
| `voronoi_pivots_siglip2_large.npy` | 1,024 KB |
| `voronoi_pivots_siglip2_large.json` | 5,727 KB |
| `voronoi_pivots_clip_large.npy` | 768 KB |
| `voronoi_pivots_clip_large.json` | 4,275 KB |

- **numpy**: `np.load()` で読み込み → shape `(256, dim)`, dtype `float32`
- **JSON**: メタデータ付き（`model_name`, `n_pivots`, `dim`, `n_images_trained`, `created_at`, `centroids`）

### 検証結果

- numpy / JSON 間の差分: **0.00e+00**（完全一致）
- 全ピボットの L2 ノルム: **1.000000**（正規化済み）
- ピボットサイズ分布: 3 モデルとも mean=91.7、median=86-90（適度に均一）

### 汎化性能の根拠

NB106 で SigLIP 2 Large について検証済み:
- Train/Test 分割: Test の MRR_ratio が Train 以上（精度劣化なし）
- 増分追加（15K → 23K）: Gini 係数 +0.005（ピボットサイズの偏り変化なし）
- ピボットとの類似度分布: Train-Test 差 -0.003（ほぼ同一）